# Diagramas de Gantt de la programación propuesta

Este cuaderno genera las figuras de la programación propuesta (mejor solución del algoritmo genético, semilla 1203) sin volver a correr el algoritmo:

- Gantt de cada etapa (troquelado, ensamble, costura y volteado)
- Ruta de una orden por las cuatro etapas (por defecto, J07)
- Uso de cada máquina y simultaneidad de los sublotes

**Archivos que hay que subir en el paso 1** (están en el proyecto o en el anexo digital):

| Archivo | Dónde está en el proyecto | Dónde está en el anexo digital |
|---|---|---|
| `final_sc_hfsagls_S2_4.py` | carpeta principal | `2_Modelo/` |
| `BD_HFS.xlsx` | carpeta principal | `1_Datos/` |
| `mejor_solucion.json` | `fase2/salida_solucion/` | `6_Solucion_propuesta/salida_solucion/` |

Ejecuta las celdas en orden con el botón ▶ (o Mayús + Enter).

## 1. Subir los tres archivos

In [ ]:
from google.colab import files

subidos = files.upload()          # selecciona los tres archivos a la vez
necesarios = {'final_sc_hfsagls_S2_4.py', 'BD_HFS.xlsx', 'mejor_solucion.json'}
faltan = necesarios - set(subidos)
print('Faltan:', ', '.join(sorted(faltan)) if faltan else 'ninguno, todo listo')

## 2. Generar las figuras

Para ver la ruta de otra orden, cambia `ORDEN` (por ejemplo, `'J09'`) y vuelve a ejecutar esta celda.

In [ ]:
import io, os, json, contextlib, importlib.util
import matplotlib.pyplot as plt

ORDEN = 'J07'                     # orden cuya ruta se dibuja

# Modelo y datos
spec = importlib.util.spec_from_file_location('modelo', 'final_sc_hfsagls_S2_4.py')
M = importlib.util.module_from_spec(spec)
spec.loader.exec_module(M)
with contextlib.redirect_stdout(io.StringIO()):
    M.cargar_datos('BD_HFS.xlsx')

# Mejor solución guardada (semilla 1203)
mejor = json.load(open('mejor_solucion.json', encoding='utf-8'))
ind = mejor['individuo']
ind = {'k': {j: int(v) for j, v in ind['k'].items()},
       'proporciones': ind['proporciones'],
       'posiciones': {int(s): [tuple(x) for x in v] for s, v in ind['posiciones'].items()}}
C, _ = M.simular_hfs(ind)
r = M.calcular_indicadores(C)
print(f"Programación: tardanza total = {r['suma_Ti']:.2f} h | makespan = {r['Cmax']:.2f} h | "
      f"órdenes tardías = {r['num_tardy']}")

# Figuras: se guardan en la carpeta graficos y se muestran abajo
plt.close('all')
os.makedirs('graficos', exist_ok=True)
figs = M.figuras_gantt(ind, orden_ruta=ORDEN, titulo=False, todas_las_rutas=True)
for nombre, fig in figs.items():
    archivo = f'gantt_{nombre}.png' if nombre.startswith('etapa') else f'{nombre}.png'
    fig.savefig(os.path.join('graficos', archivo), dpi=200, bbox_inches='tight')
    print('Guardada:', archivo)
plt.show()

## 3. Descargar las figuras (archivo .zip)

In [ ]:
import shutil
shutil.make_archive('graficos', 'zip', 'graficos')
files.download('graficos.zip')